# Lab 4 — Дообучение вокодера kNN-VC (пункт 2.1)

Данный ноутбук дообучает HiFi-GAN-вокодер из [bshall/knn-vc](https://github.com/bshall/knn-vc) на одном целевом голосе и сохраняет веса в `finetuned_model.pt`. Полученный файл затем подключается в Lab4Panel через поле «Дообученный вокодер (.pt)» или через флаг `--vocoder-ckpt` в `lab4_vc.py`.

## Инструкция

1. Открой этот .ipynb в [Google Colab](https://colab.research.google.com/) (Runtime → Change runtime type → GPU).
2. Загрузи правым кликом в панель файлов Colab файл `target_voice.wav` (речь целевого голоса, желательно от 30 секунд).
3. Выполни ячейки по порядку.
4. В конце файл `finetuned_model.pt` скачается автоматически. Положи его в `python/` вашего проекта и укажи путь в поле GUI.

## 0. Проверка GPU

In [ ]:
!nvidia-smi

## 1. Зависимости

kNN-VC требует `torch >= 2.0`, `torchaudio`, `numpy`. В Colab это всё уже есть, достаточно проверить.

In [ ]:
import torch, torchaudio
print('torch =', torch.__version__, '| cuda =', torch.cuda.is_available())
print('torchaudio =', torchaudio.__version__)

## 2. Загрузка базовой модели kNN-VC

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
from torch.amp import autocast, GradScaler
import os
import random
from tqdm import tqdm

print('Загрузка базовой модели kNN-VC...')
knn_vc = torch.hub.load('bshall/knn-vc:master', 'knn_vc',
                        trust_repo=True, pretrained=True, device='cuda')

wavlm = knn_vc.wavlm
vocoder = knn_vc.hifigan
vocoder.eval()

## 3. Параметры дообучения

In [ ]:
AUDIO_FILE = 'target_voice.wav'
EPOCHS = 80
CHUNK_DURATION = 2.0
SAMPLE_RATE = 16000
HOP_RATIO = 320

optimizer = optim.AdamW(vocoder.parameters(), lr=1e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = GradScaler('cuda')

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000, n_fft=1024, win_length=1024, hop_length=256,
    n_mels=80, f_min=0, f_max=8000,
).to('cuda')

def combined_loss(fake_audio, real_audio, alpha=0.1):
    l1 = nn.functional.l1_loss(fake_audio, real_audio)
    fake_mel = mel_transform(fake_audio + 1e-6)
    real_mel = mel_transform(real_audio + 1e-6)
    mel_loss = nn.functional.l1_loss(
        torch.log(fake_mel + 1e-5), torch.log(real_mel + 1e-5),
    )
    return l1 + alpha * mel_loss, l1.item(), mel_loss.item()

## 4. Загрузка целевого голоса и извлечение WavLM-фич (слой 6)

In [ ]:
if not os.path.exists(AUDIO_FILE):
    raise FileNotFoundError(f'Файл {AUDIO_FILE} не найден. Загрузи target_voice.wav в панель файлов Colab.')

print(f'Загрузка аудио: {AUDIO_FILE}')
waveform, sr = torchaudio.load(AUDIO_FILE)
if sr != 16000:
    waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
waveform = waveform.to('cuda')
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)
waveform = waveform / (waveform.abs().max() + 1e-5)

print('Извлечение WavLM-фич (слой 6)...')
with torch.no_grad():
    wavlm_features = wavlm.extract_features(
        waveform, output_layer=6, ret_layer_results=False,
    )[0]
    print(f'Форма фич: {tuple(wavlm_features.shape)}')

total_steps = wavlm_features.shape[1]
val_steps = max(20, int(total_steps * 0.2))
train_steps = total_steps - val_steps
print(f'Train: {train_steps} шагов, Val: {val_steps} шагов')

chunk_time_samples = int(CHUNK_DURATION * SAMPLE_RATE)
chunk_feature_steps = chunk_time_samples // HOP_RATIO
val_start_idx = train_steps

## 5. Цикл дообучения

In [ ]:
criterion = combined_loss
print(f'Start Fine-Tuning ({EPOCHS} эпох), чанки по {CHUNK_DURATION} сек...')

best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 15

for epoch in tqdm(range(EPOCHS)):
    optimizer.zero_grad()
    vocoder.train()
    start_idx = random.randint(0, max(0, train_steps - chunk_feature_steps - 1))
    feat_chunk = wavlm_features[:, start_idx:start_idx + chunk_feature_steps, :]
    audio_start = start_idx * HOP_RATIO
    audio_end = audio_start + chunk_feature_steps * HOP_RATIO
    waveform_chunk = waveform[:, audio_start:audio_end]

    with autocast('cuda'):
        fake_audio = vocoder(feat_chunk)
        if fake_audio.dim() == 3 and fake_audio.shape[1] == 1:
            fake_audio = fake_audio.squeeze(1)
        min_len = min(waveform_chunk.shape[1], fake_audio.shape[1])
        loss, l1_val, mel_val = criterion(
            fake_audio[:, :min_len], waveform_chunk[:, :min_len],
        )

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(vocoder.parameters(), max_norm=1.0)
    scaler.step(optimizer)
    scheduler.step()
    scaler.update()

    if epoch % 10 == 0 or epoch == EPOCHS - 1:
        vocoder.eval()
        with torch.no_grad():
            val_feat = wavlm_features[:, val_start_idx:val_start_idx + chunk_feature_steps, :]
            val_audio_start = val_start_idx * HOP_RATIO
            val_audio_end = val_audio_start + chunk_feature_steps * HOP_RATIO
            val_waveform = waveform[:, val_audio_start:val_audio_end]
            with autocast('cuda'):
                val_fake = vocoder(val_feat)
                if val_fake.dim() == 3 and val_fake.shape[1] == 1:
                    val_fake = val_fake.squeeze(1)
                min_len = min(val_waveform.shape[1], val_fake.shape[1])
                val_loss, _, _ = criterion(
                    val_fake[:, :min_len], val_waveform[:, :min_len],
                )
            val_loss_val = val_loss.item()
            print(f'\nEpoch {epoch}: Train Loss={loss.item():.4f} | Val Loss={val_loss_val:.4f}')
            if val_loss_val < best_val_loss:
                best_val_loss = val_loss_val
                patience_counter = 0
                torch.save(vocoder.state_dict(), 'finetuned_model_best.pt')
                print(f'Новая лучшая модель сохранена. Val Loss={best_val_loss:.4f}')
            else:
                patience_counter += 1
                print(f'Нет улучшений: {patience_counter}/{EARLY_STOP_PATIENCE}')
            if patience_counter >= EARLY_STOP_PATIENCE:
                print('Early stopping: валидация не улучшается')
                break

    if epoch % 20 == 0:
        torch.cuda.empty_cache()
        print(f'Epoch {epoch}: L1={l1_val:.4f}, Mel={mel_val:.4f}, LR={optimizer.param_groups[0]["lr"]:.2e}')

## 6. Сохранение и скачивание весов

In [ ]:
if os.path.exists('finetuned_model_best.pt'):
    vocoder.load_state_dict(torch.load('finetuned_model_best.pt', map_location='cuda'))
save_path = 'finetuned_model.pt'
torch.save(vocoder.state_dict(), save_path)

print('Генерация тестового сэмпла...')
vocoder.eval()
with torch.no_grad():
    test_start = random.randint(0, max(0, total_steps - chunk_feature_steps - 1))
    test_feat = wavlm_features[:, test_start:test_start + chunk_feature_steps, :]
    test_audio = vocoder(test_feat)
    if test_audio.dim() == 3 and test_audio.shape[1] == 1:
        test_audio = test_audio.squeeze(1)
    torchaudio.save('test_output.wav', test_audio.cpu(), 16000)

from google.colab import files
files.download(save_path)